# Life Quality Predictor - Pipeline Training

This notebook trains a Random Forest model with a scikit-learn Pipeline and generates a ydata-profiling report.

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from ydata_profiling import ProfileReport
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

print("Setup completed successfully.")

ModuleNotFoundError: No module named 'ydata_profiling'

In [ ]:

csv_path = 'dataset.csv'
if not os.path.exists(csv_path):
    csv_path = '../dataset.csv'

data = pd.read_csv(csv_path)
print(f"Loaded dataset with shape: {data.shape}")
print(data.head())

In [ ]:

profile = ProfileReport(data, title="City Livability Data Profiling Report")
profile.to_file("data_profile_report.html")
print("Profiling report saved as data_profile_report.html")

In [ ]:

target_col = 'livability_score'
numeric_features = [
    'aqi', 'temperature', 'humidity', 'green_cover', 
    'hospital_density', 'school_density', 'internet_score', 
    'population_density', 'crime_rate', 
    'employment_rate', 'cost_of_living'
]
nominal_features = ['climate_zone']
ordinal_features = ['development_tier']

num_cols = [c for c in numeric_features if c in data.columns]
nom_cols = [c for c in nominal_features if c in data.columns]
ord_cols = [c for c in ordinal_features if c in data.columns]

data_clean = data.dropna(subset=[target_col])
X = data_clean[num_cols + nom_cols + ord_cols]
y = data_clean[target_col].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('knn_imputer', KNNImputer(n_neighbors=3)),
    ('scaler', StandardScaler())
])

nom_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

ord_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder(categories=[['Tier 3', 'Tier 2', 'Tier 1']], handle_unknown='use_encoded_value', unknown_value=-1))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_cols),
    ('nom', nom_transformer, nom_cols),
    ('ord', ord_transformer, ord_cols)
])


pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=50, max_depth=4, random_state=42))
])

print("Pipeline constructed successfully.")

In [ ]:

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Test MSE: {mse:.4f}")
print(f"Test R2: {r2:.4f}")

In [ ]:

importances = pipeline.named_steps['regressor'].feature_importances_
feature_names_out = pipeline.named_steps['preprocessor'].get_feature_names_out()


cleaned_feature_names = []
for name in feature_names_out:
    cleaned_name = name
    for prefix in ['num__', 'nom__', 'ord__']:
        if cleaned_name.startswith(prefix):
            cleaned_name = cleaned_name[len(prefix):]
            break
    cleaned_feature_names.append(cleaned_name)

feature_importances = pd.Series(importances, index=cleaned_feature_names).sort_values(ascending=True)

plt.figure(figsize=(10, 6))
feature_importances.plot(kind='barh', color='teal')
plt.title('Feature Importances')
plt.xlabel('Importance Value')
plt.tight_layout()
plt.show()

In [ ]:


pipeline.fit(X, y)
y_full_pred = pipeline.predict(X)
full_mse = mean_squared_error(y, y_full_pred)
full_r2 = r2_score(y, y_full_pred)

feature_importances_dict = {name: float(imp) for name, imp in zip(cleaned_feature_names, importances)}
feature_importances_dict = dict(sorted(feature_importances_dict.items(), key=lambda item: item[1], reverse=True))

bundle = {
    "pipeline": pipeline,
    "feature_names": num_cols + nom_cols + ord_cols,
    "metrics": {
        "mse": float(full_mse),
        "r2": float(full_r2),
        "samples_count": len(y)
    },
    "feature_importances": feature_importances_dict
}

joblib.dump(bundle, '../livability_model.pkl')
print("Model bundle persisted to '../livability_model.pkl'")